In [13]:
from dotenv import load_dotenv
from langgraph.types import interrupt, Command
import takehomeV2
load_dotenv()



True

In [23]:
examples = [
    {
        "inputs": {"severity": 1},
        "outputs": {"expected_route": "assess_case"},
    },
    {
        "inputs": {"severity": 2},
        "outputs": {"expected_route": "acknowledge_case"},
    },
    {
        "inputs": {"severity": 3},
        "outputs": {"expected_route": "acknowledge_case"},
    },
    {
        "inputs": {"severity": 4},
        "outputs": {"expected_route": "acknowledge_case"},
    },
    {
        "inputs": {"severity": 5},
        "outputs": {"expected_route": "validateSeverity"},
    },
    {
        "inputs": {"severity": 0},
        "outputs": {"expected_route": "validateSeverity"},
    },
    {
        "inputs": {"severity": "1"},
        "outputs": {"expected_route": "validateSeverity"},
    },
    {
        "inputs": {"severity": None},
        "outputs": {"expected_route": "validateSeverity"},
    },
]

In [24]:
from langsmith import Client 
client = Client()
dataset = client.create_dataset(
    dataset_name="Severity Validation",
    description="Check valid and invalid severity routing.",
)

client.create_examples(
    dataset_id=dataset.id,
    examples=examples,
)

{'example_ids': ['2cd1020e-5ef6-4d16-abf5-3d5d7cdd36d9',
  '71ef8b57-0204-404a-bdd8-39a7ec033e12',
  '037fce23-43ed-4e33-8547-6b3f495eb915',
  'd0c88bef-c9da-42e2-936a-2b7069c10d61',
  'db3c9c8a-3780-45d1-b63f-562fab4448b3',
  '4e606d0e-1bf3-4e76-a995-be2cf43c3c0b',
  'af97aa2f-4e8e-4441-82a7-d7bef2816a99',
  '9de627d3-0627-4272-91f8-c0c0d0b6fd60'],
 'count': 8,
 'as_of': '2026-11-04T04:13:01.170786981Z'}

In [25]:
def severity_target(inputs):
    route = takehomeV2.route_case(inputs)

    return {"actual_route": route}

In [26]:
def severity_evaluator(outputs, reference_outputs):
    actual = outputs["actual_route"]
    expected = reference_outputs["expected_route"]

    return {
        "key": "severity_validation",
        "score": int(actual == expected),
        "comment": f"Expected: {expected}. Actual: {actual}.",
    }

In [27]:
results = client.evaluate(
    severity_target,
    data="Severity Validation",
    evaluators=[severity_evaluator],
    experiment_prefix="severity-validation",
    max_concurrency=1,
)

View the evaluation results for experiment: 'severity-validation-ad3eb871' at:
https://smith.langchain.com/o/edf925e8-923e-4a29-86cd-32644621291e/datasets/970340b4-e938-4699-8f7a-8f69039b04dc/compare?selectedSessions=e6f2f990-1788-40a9-9e9d-c2120856aef5




In [28]:
for example in examples:
    actual = severity_target(example["inputs"])
    feedback = severity_evaluator(actual, example["outputs"])
    print(feedback)

    assert feedback["score"] == 1, feedback["comment"]

print("All severity validation checks passed.")

{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: assess_case. Actual: assess_case.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: acknowledge_case. Actual: acknowledge_case.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: acknowledge_case. Actual: acknowledge_case.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: acknowledge_case. Actual: acknowledge_case.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: validateSeverity. Actual: validateSeverity.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: validateSeverity. Actual: validateSeverity.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: validateSeverity. Actual: validateSeverity.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: validateSeverity. Actual: validateSeverity.'}
All severity validation checks passed.
